Please use this notebook to perform any initial analyses and model testing!

In [0]:
!pip install kagglehub
!pip install --force-reinstall --no-deps /Volumes/workspace/ds_cb_wheels/wheels/eda_toolkit-0.1.0-py3-none-any.whl

In [0]:
dbutils.library.restartPython()

In [0]:
from eda_toolkit.first_stage_report_building.source.multicolinearity import test_multicollinearity

In [0]:
import pandas as pd
import numpy as np
import os 
import scipy
import kagglehub
from eda_toolkit.first_stage_report_building.ops.generate_report import build_report_regression

In [0]:
def collect_from_kaggle(kaggle_path: str) -> pd.DataFrame:
    path = kagglehub.dataset_download(kaggle_path)
    file_dir = os.listdir(path)
    df_path = f"{path}/{file_dir[0]}"
    df = pd.read_csv(df_path)

    return df

df_bronze = collect_from_kaggle("berkerisen/wind-turbine-scada-dataset")#
df_bronze.to_csv("../data/bronze/bronze.csv")
df_bronze.head()

In [0]:
df_bronze.columns = ['timestamp', 'lv_active_power', 'wind_speed', 'theoretical_power_curve', 'wind_direction']
df_bronze.head()

In [0]:
print(df_bronze.columns)

In [0]:
df_bronze['timestamp'] = pd.to_datetime(df_bronze['timestamp'], format='%d %m %Y %H:%M')
print(df_bronze[df_bronze['timestamp'] == pd.NaT])
df_bronze.head()

In [0]:
test_multicollinearity(df_bronze, ['lv_active_power', 'wind_speed', 'theoretical_power_curve', 'wind_direction'], 'lv_active_power')

In [0]:
build_report_regression(df_bronze, 'lv_active_power', ['timestamp'])

In [0]:
print(df_bronze.loc[df_bronze['lv_active_power'] < 0]['lv_active_power'].describe())

In [0]:
df_bronze['lv_active_power'] = df_bronze['lv_active_power'].clip(lower = 0)

In [0]:
df_bronze['hour'] = pd.to_datetime(df_bronze['timestamp']).dt.hour
df_bronze['monr'] = pd.to_datetime(df_bronze['timestamp']).dt.hour

In [0]:
print(df_bronze['timestamp'].min())
print(df_bronze['timestamp'].max())

In [0]:
df_bronze.to_csv("../data/silver/silver.csv")

In [0]:
df_bronze.head()

In [0]:
df_bronze['sine_hour'] = np.sin(2 * np.pi * df_bronze['hour']/24)
df_bronze['cosine_hour'] = np.cos(2 * np.pi * df_bronze['hour']/24)

df_bronze = df_bronze.sort_values(by = ['timestamp'])

In [0]:
df_bronze['wind_direction_rad'] = np.deg2rad(df_bronze['wind_direction'])
df_bronze['wind_magnitude_i'] = df_bronze['wind_speed'] * (np.cos(df_bronze['wind_direction_rad']))
df_bronze['wind_magnitude_j'] = df_bronze['wind_speed'] * (np.sin(df_bronze['wind_direction_rad']))

df_bronze.to_csv("../data/gold/gold.csv")

In [0]:
df_bronze['wind_magnitude_i_3h_rolling'] = df_bronze['wind_magnitude_i'].rolling(window = 3).mean()
df_bronze['wind_magnitude_j_3h_rolling'] = df_bronze['wind_magnitude_j'].rolling(window = 3).mean()
df_bronze['wind_speed_3h_rolling'] = df_bronze['wind_speed'].rolling(window = 3).mean()
df_bronze.dropna(inplace=True)

In [0]:
from datetime import datetime

train_month_threshold = datetime(2018, 3, 1, 0, 0, 0, 0)
val_month_threshold = datetime(2018, 4, 1, 0, 0, 0, 0)
test_month_threshold = datetime(2018, 8, 1, 0, 0, 0, 0)

feature_cols = ['wind_speed', 'wind_direction', 'sine_hour', 'cosine_hour', 'wind_magnitude_i', 'wind_magnitude_j', 'wind_magnitude_i_3h_rolling', 'wind_magnitude_j_3h_rolling', 'wind_speed_3h_rolling']
target_col = ['lv_active_power']

df_train = df_bronze.loc[df_bronze['timestamp'] < train_month_threshold]
df_val = df_bronze.loc[(df_bronze['timestamp'] >= train_month_threshold) & (df_bronze['timestamp'] <= val_month_threshold)]

In [0]:
X_train = df_train[feature_cols]
y_train = df_train[target_col]

X_val = df_val[feature_cols]
y_val = df_val[target_col]
t_val = df_val['timestamp']

print(len(X_train))
print(len(X_val))

In [0]:
from sklearn.ensemble import RandomForestRegressor

rf_params = {
    "n_estimators": 300,        # enough trees to stabilize predictions without excessive runtime
    "max_depth": 15,            # caps tree complexity; prevents overfitting on ~50k rows with few features
    "min_samples_split": 10,    # requires a reasonable sample size before splitting further
    "min_samples_leaf": 5,      # avoids leaves fit to tiny, noisy subsets
    "max_features": "sqrt",     # standard default for RF; decorrelates trees
    "n_jobs": -1,                # use all available cores
    "random_state": 42,          # reproducibility
}

rf = RandomForestRegressor(**rf_params)

rf.fit(X_train, y_train)

test_y_hat = rf.predict(X_val)

predictions_set = X_val.copy()

predictions_set['y'] = y_val
predictions_set['y_hat'] = test_y_hat

predictions_set

In [0]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

first_val_mae = mean_absolute_error(y_val, test_y_hat)
print(first_val_mae)

In [0]:
first_val_rmse = root_mean_squared_error(y_val, test_y_hat)

print(first_val_rmse)

In [0]:
first_val_r2 = r2_score(y_val, test_y_hat)

print(first_val_r2)

In [0]:
predictions_set['residual'] = predictions_set['y'] - predictions_set['y_hat']

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize = (12, 8))
_ = sns.scatterplot(data = predictions_set, x = 'wind_speed', y = 'residual')

In [0]:
predictions_set['timestamp'] = t_val

negative_resid = predictions_set.loc[predictions_set['residual'] <= -1000]

print(negative_resid['timestamp'].min())
print(negative_resid['timestamp'].max())

In [0]:
for timestamp in negative_resid['timestamp']:
    print(timestamp)

In [0]:
from datetime import timedelta

maes = []
rmses = []
r2s = []
percs = []
for weeks in range(1,10):
    train_month_range = train_month_threshold + timedelta(days = weeks*7)
    val_month_range = val_month_threshold + timedelta(days = weeks*7)

    df_train = df_bronze.loc[df_bronze['timestamp'] < train_month_range]
    df_val = df_bronze.loc[(df_bronze['timestamp'] >= train_month_range) & (df_bronze['timestamp'] <= val_month_range)]

    X_train = df_train[feature_cols]
    y_train = df_train[target_col]

    X_val = df_val[feature_cols]
    y_val = df_val[target_col]

    rf = RandomForestRegressor(**rf_params)

    rf.fit(X_train, y_train)

    test_y_hat = rf.predict(X_val)

    predictions_set = X_val.copy()

    predictions_set['y'] = y_val
    predictions_set['y_hat'] = test_y_hat

    first_val_mae = mean_absolute_error(y_val, test_y_hat)
    first_val_rmse = root_mean_squared_error(y_val, test_y_hat)
    first_val_r2 = r2_score(y_val, test_y_hat)

    maes.append(first_val_mae)
    rmses.append(first_val_rmse)
    r2s.append(first_val_r2)

    df_190 = predictions_set.loc[predictions_set['wind_direction'].between(190, 240)]
    percs.append(len(df_190)/len(predictions_set))


print(maes)
print(rmses)
print(r2s)
print(percs)
    